<div dir="rtl">

# 🧠 05 - Chat History & Memory Management (تجارب إدارة الذاكرة وسجل المحادثات)

## ما هو هذا الكراس؟
يقدم هذا الكراس مساراً تجريبياً ومنظماً لاستكشاف وتقنيات **إدارة سجل المحادثات والذاكرة (Chat History & Memory)** في LangChain باستخدام المعمارية الحديثة `RunnableWithMessageHistory` و `trim_messages`.

## المحاور الرئيسية التي يتم تناولها:
1. **أنواع الرسائل القياسية**: استخدام `HumanMessage` و `AIMessage` لتمثيل طرفي المحادثة.
2. **مخزن الذاكرة والجلسات المستقلة**: إنشاء مخزن جلسات بـ `ChatMessageHistory` والربط بـ `RunnableWithMessageHistory`.
3. **قوالب التوجيه التفاعلية**: دمج `MessagesPlaceholder` والتخصيص اللغوي الديناميكي (`{language}`).
4. **قص وتقليم الرسائل القديمة (Message Trimming)**: التحكم بحجم التوكنز ومنع طفح نافذة السياق باستغلال `trim_messages`.

</div>

### 1️⃣ تحميل البيئة وتهيئة نموذج المحادثة (ChatGroq)

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())

# تهيئة نموذج Groq
model = ChatGroq(
    api_key=os.environ.get("GROQ_API_KEY"),
    model="openai/gpt-oss-20b"
)

print("✅ تم تحميل البيئة وتهيئة النموذج بنجاح.")

### 2️⃣ التفاعل المباشر مع الرسائل الحرة (`HumanMessage` & `AIMessage`)

In [ ]:
# إرسال قائمة من الرسائل التتابعية لتزويد النموذج بسياق مباشر
messages = [
    HumanMessage(content="Hi, My name is Ali. I learned about you from the LangChain documentation."),
    AIMessage(content="Hello Ali! I'm an AI language model here to assist you with LangChain."),
    HumanMessage(content="Can you tell me what is my name?")
]

response = model.invoke(messages)
print("🤖 إجابة النموذج بناءً على السياق المباشر:")
print(response.content)

### 3️⃣ إدارة الذاكرة والجلسات المستقلة (`RunnableWithMessageHistory`)

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# قاموس محلي لحفظ تاريخ الرسائل لكل session_id
store = {}

def get_history(key: str) -> BaseChatMessageHistory:
    """إرجاع أو إنشاء سجل المحادثة الخاص بالجلسة المحددة"""
    if key not in store:
        store[key] = ChatMessageHistory()
    return store[key]

# ربط النموذج بمدير الذاكرة والجلسات
with_history = RunnableWithMessageHistory(model, get_history)

# 1. اختبار الجلسة الأولى (my_session_1)
config_1 = {"configurable": {"session_id": "my_session_1"}}
with_history.invoke([HumanMessage(content="Hi, My name is Ali.")], config=config_1)
res1 = with_history.invoke([HumanMessage(content="What is my name?")], config=config_1)
print("📌 الجلسة الأولى (Session 1):", res1.content)

# 2. اختبار معزول للجلسة الثانية (my_session_2)
config_2 = {"configurable": {"session_id": "my_session_2"}}
res2 = with_history.invoke([HumanMessage(content="What is my name?")], config=config_2)
print("📌 الجلسة الثانية المعزولة (Session 2):", res2.content)

### 4️⃣ قوالب التوجيه بـ MessagesPlaceholder والتخصيص اللغوي ({language})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# إنشاء قالب توجيه ديناميكي يدعم سجل الرسائل ولغات متعددة
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Answer the question based on conversation history in {language}."
    ),
    MessagesPlaceholder(variable_name="messages"),
    ("human", "{question}"),
])

chain = prompt | model

# ربط السلسلة الهيكلية بالذاكرة وتحديد المפתاح اللغوي وحقل الأسئلة
with_message_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="question",
    history_messages_key="messages"
)

config_arabic = {"configurable": {"session_id": "my_session_arabic"}}

response = with_message_history.invoke(
    {
        "language": "Arabic",
        "question": "مرحباً، أنا اسمي علي وأدرس LangChain."
    },
    config=config_arabic
)

print("🤖 الاستجابة التفاعلية باللغة العربية:")
print(response.content)

### 5️⃣ تقليم وقص الرسائل القديمة (Message Trimming) بـ `trim_messages`

In [ ]:
from operator import itemgetter
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

# إنشاء أداة تقليم الرسائل للحفاظ على أحدث الرسائل فقط وفق حد التوكنز
trimmed_messages = trim_messages(
    max_tokens=200,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
)

# بناء سلسلة LCEL تقص الرسائل القديمة تلقائياً وتمرر الحديثة للتوجيه
trim_chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmed_messages)
    | prompt
    | model
)

# تجربة تقليم قائمة طويلة من الرسائل
conversation_history = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, My name is Ali. I learned about you from the LangChain documentation."),
    AIMessage(content="Hello Ali! I'm an AI language model here to assist you."),
    HumanMessage(content="Can you tell me what is my name?"),
    AIMessage(content="Your name is Ali."),
]

res_trimmed = trim_chain.invoke({
    "language": "Arabic",
    "messages": conversation_history,
    "question": "ما هو اسمي وماذا أفعل؟"
})

print("✂️ إجابة النموذج بعد تقليم المحادثة القديمة:")
print(res_trimmed.content)

<div dir="rtl">

## 💡 الخلاصة العملية:
- تم تنظيف وترتيب كراس التجارب بشكل منطقي وتدريجي.
- يغطي الكراس الآن البداية المباشرة مع `HumanMessage` وحتى التقنيات المتقدمة لتقليم الرسائل (`trim_messages`).
- يمكنك الاستمرار في إضافة أفكارك وتجاربك داخل هذا المجلد في أي وقت!

</div>